# PPO on a Stochastic Gridworld
## How local neighborhood view size affects generalization

Each episode, obstacles are **randomly regenerated** — so the agent can't memorize paths.
It must learn a generalizable navigation policy from its **local view** alone.

We compare view radii: `r=1` (3×3 patch), `r=2` (5×5), `r=3` (7×7), and `r=full` (whole grid).


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import matplotlib.gridspec as gridspec
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
print(f'PyTorch {torch.__version__}')

## 1. Stochastic Gridworld Environment

- **Grid**: 10×10, fixed start (top-left) and goal (bottom-right)
- **Obstacles**: 20% of non-start/goal cells randomly filled each episode
- **State**: local neighborhood view (NxN patch) centered on agent, padded with walls at edges
- **Reward**: +1.0 reach goal, -0.01 per step, -0.2 hit wall

In [ ]:
class StochasticGridworld:
    """
    10x10 grid with randomly regenerated obstacles each episode.
    Agent starts at (0,0), goal at (9,9).
    State is a local neighborhood view of radius `view_radius`.
    """
    EMPTY, WALL, AGENT, GOAL = 0, 1, 2, 3
    ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]  # up, down, left, right
    ACTION_NAMES = ['↑','↓','←','→']

    def __init__(self, grid_size=10, obstacle_density=0.20, view_radius=1, max_steps=200):
        self.G = grid_size
        self.density = obstacle_density
        self.view_radius = view_radius
        self.max_steps = max_steps
        self.start = (0, 0)
        self.goal  = (self.G-1, self.G-1)
        self.grid = None
        self.pos  = None
        self.steps = 0

    @property
    def obs_size(self):
        if self.view_radius == -1:  # full grid
            return self.G * self.G
        side = 2 * self.view_radius + 1
        return side * side

    def _make_grid(self):
        """Generate a new random obstacle layout, guarantee a path exists."""
        for _ in range(100):  # retry until solvable
            grid = np.zeros((self.G, self.G), dtype=np.int8)
            cells = [(r,c) for r in range(self.G) for c in range(self.G)
                     if (r,c) != self.start and (r,c) != self.goal]
            n_walls = int(self.density * len(cells))
            walls = random.sample(cells, n_walls)
            for r, c in walls:
                grid[r, c] = self.WALL
            if self._bfs(grid, self.start, self.goal):
                return grid
        return np.zeros((self.G, self.G), dtype=np.int8)  # fallback: empty

    def _bfs(self, grid, start, goal):
        """Check if path exists from start to goal."""
        visited = set([start])
        queue = deque([start])
        while queue:
            r, c = queue.popleft()
            if (r, c) == goal:
                return True
            for dr, dc in self.ACTIONS:
                nr, nc = r+dr, c+dc
                if 0<=nr<self.G and 0<=nc<self.G and grid[nr,nc]!=self.WALL and (nr,nc) not in visited:
                    visited.add((nr,nc))
                    queue.append((nr,nc))
        return False

    def reset(self):
        self.grid = self._make_grid()
        self.pos   = self.start
        self.steps = 0
        return self._get_obs()

    def step(self, action):
        dr, dc = self.ACTIONS[action]
        nr, nc = self.pos[0]+dr, self.pos[1]+dc
        self.steps += 1
        hit_wall = False
        if 0<=nr<self.G and 0<=nc<self.G and self.grid[nr,nc]!=self.WALL:
            self.pos = (nr, nc)
        else:
            hit_wall = True

        done = False
        if self.pos == self.goal:
            reward = 1.0
            done = True
        elif self.steps >= self.max_steps:
            reward = -0.5
            done = True
        else:
            reward = -0.01 + (-0.2 if hit_wall else 0.0)

        return self._get_obs(), reward, done

    def _get_obs(self):
        if self.view_radius == -1:
            # Full grid view: 0=empty, 1=wall, 0.5=goal
            obs = self.grid.astype(np.float32).copy()
            obs[self.goal] = 0.5
            return obs.flatten()

        r, c = self.pos
        rad = self.view_radius
        side = 2*rad+1
        patch = np.ones((side, side), dtype=np.float32)  # default: wall (out of bounds)
        for dr in range(-rad, rad+1):
            for dc in range(-rad, rad+1):
                nr, nc = r+dr, c+dc
                if 0<=nr<self.G and 0<=nc<self.G:
                    if (nr, nc) == self.goal:
                        patch[dr+rad, dc+rad] = 0.5   # goal marker
                    else:
                        patch[dr+rad, dc+rad] = float(self.grid[nr, nc])
        return patch.flatten()

    def render_grid(self):
        """Return a 2D array for visualization."""
        vis = self.grid.astype(np.float32).copy()
        vis[self.goal]  = 3
        vis[self.pos]   = 2
        vis[self.start] = 4 if self.pos != self.start else 2
        return vis

# Quick sanity check
env = StochasticGridworld(view_radius=1)
obs = env.reset()
print(f'obs_size (r=1): {env.obs_size}  →  shape {obs.shape}')
env2 = StochasticGridworld(view_radius=2)
print(f'obs_size (r=2): {env2.obs_size}')
env3 = StochasticGridworld(view_radius=-1)
print(f'obs_size (full): {env3.obs_size}')

## 2. PPO Agent

Standard PPO with:
- Shared MLP backbone → separate policy (actor) and value (critic) heads
- GAE (Generalized Advantage Estimation)
- Clipped surrogate objective
- Entropy bonus to encourage exploration

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_size, n_actions=4, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_size, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, n_actions)
        self.critic = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

    def get_action(self, obs):
        logits, value = self(obs)
        dist  = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action.item(), dist.log_prob(action), value.squeeze(), dist.entropy()


class PPOAgent:
    def __init__(self, obs_size, n_actions=4, lr=3e-4, gamma=0.99, gae_lambda=0.95,
                 clip_eps=0.2, entropy_coef=0.01, vf_coef=0.5,
                 n_steps=512, n_epochs=4, batch_size=64):
        self.gamma      = gamma
        self.gae_lambda = gae_lambda
        self.clip_eps   = clip_eps
        self.ent_coef   = entropy_coef
        self.vf_coef    = vf_coef
        self.n_steps    = n_steps
        self.n_epochs   = n_epochs
        self.batch_size = batch_size

        self.net = ActorCritic(obs_size, n_actions)
        self.opt = optim.Adam(self.net.parameters(), lr=lr)

        # Rollout buffers
        self.obs_buf, self.act_buf, self.logp_buf = [], [], []
        self.rew_buf, self.val_buf, self.done_buf  = [], [], []

    def collect_rollout(self, env):
        """Collect n_steps of experience."""
        self.obs_buf.clear(); self.act_buf.clear(); self.logp_buf.clear()
        self.rew_buf.clear(); self.val_buf.clear(); self.done_buf.clear()

        obs = env.reset()
        ep_returns, ep_lengths = [], []
        ep_ret, ep_len = 0, 0

        for _ in range(self.n_steps):
            obs_t = torch.FloatTensor(obs).unsqueeze(0)
            with torch.no_grad():
                action, logp, value, _ = self.net.get_action(obs_t)

            next_obs, reward, done = env.step(action)

            self.obs_buf.append(obs)
            self.act_buf.append(action)
            self.logp_buf.append(logp.item())
            self.rew_buf.append(reward)
            self.val_buf.append(value.item())
            self.done_buf.append(done)

            ep_ret += reward; ep_len += 1
            obs = next_obs
            if done:
                ep_returns.append(ep_ret); ep_lengths.append(ep_len)
                ep_ret, ep_len = 0, 0
                obs = env.reset()

        # Bootstrap last value
        with torch.no_grad():
            _, last_val = self.net(torch.FloatTensor(obs).unsqueeze(0))
        return ep_returns, ep_lengths, last_val.item()

    def _compute_gae(self, last_val):
        advantages = np.zeros(self.n_steps, dtype=np.float32)
        gae = 0
        for t in reversed(range(self.n_steps)):
            next_val  = last_val if t == self.n_steps-1 else self.val_buf[t+1]
            next_done = 0 if t == self.n_steps-1 else float(self.done_buf[t+1])
            delta     = self.rew_buf[t] + self.gamma * next_val * (1 - next_done) - self.val_buf[t]
            gae = delta + self.gamma * self.gae_lambda * (1 - next_done) * gae
            advantages[t] = gae
        returns = advantages + np.array(self.val_buf, dtype=np.float32)
        return advantages, returns

    def update(self, last_val):
        advantages, returns = self._compute_gae(last_val)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        obs_t  = torch.FloatTensor(np.array(self.obs_buf))
        act_t  = torch.LongTensor(self.act_buf)
        logp_t = torch.FloatTensor(self.logp_buf)
        adv_t  = torch.FloatTensor(advantages)
        ret_t  = torch.FloatTensor(returns)

        total_loss = 0
        n = len(obs_t)
        for _ in range(self.n_epochs):
            idx = torch.randperm(n)
            for start in range(0, n, self.batch_size):
                mb = idx[start:start+self.batch_size]
                logits, values = self.net(obs_t[mb])
                dist     = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(act_t[mb])
                entropy  = dist.entropy().mean()

                ratio    = torch.exp(new_logp - logp_t[mb])
                s1       = ratio * adv_t[mb]
                s2       = torch.clamp(ratio, 1-self.clip_eps, 1+self.clip_eps) * adv_t[mb]
                pg_loss  = -torch.min(s1, s2).mean()
                vf_loss  = 0.5 * (values.squeeze() - ret_t[mb]).pow(2).mean()
                loss     = pg_loss + self.vf_coef * vf_loss - self.ent_coef * entropy

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
                self.opt.step()
                total_loss += loss.item()
        return total_loss

print('PPO agent defined ✓')

## 3. Training Loop

Train PPO for each view radius and record learning curves.
This cell takes **3–5 minutes** depending on your hardware.

In [ ]:
def train_ppo(view_radius, total_steps=200_000, seed=42):
    """
    Train a PPO agent on the stochastic gridworld with a given view radius.
    view_radius = -1 means full grid observation.
    Returns dict of training statistics.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    env   = StochasticGridworld(view_radius=view_radius)
    agent = PPOAgent(obs_size=env.obs_size)

    steps_done = 0
    all_returns, all_steps = [], []
    success_window = deque(maxlen=50)

    label = f'r={view_radius}' if view_radius != -1 else 'full'
    print(f'Training {label}  (obs_size={env.obs_size})', end='', flush=True)

    while steps_done < total_steps:
        ep_returns, _, last_val = agent.collect_rollout(env)
        agent.update(last_val)
        steps_done += agent.n_steps

        for r in ep_returns:
            all_returns.append(r)
            all_steps.append(steps_done)
            success_window.append(1 if r > 0.5 else 0)

        if steps_done % 20_000 < agent.n_steps:
            sr = np.mean(success_window) if success_window else 0
            print(f'.', end='', flush=True)

    print(f'  done. Final success rate: {np.mean(list(success_window)):.1%}')
    return {
        'label':      label,
        'view_radius': view_radius,
        'obs_size':    env.obs_size,
        'steps':       all_steps,
        'returns':     all_returns,
        'agent':       agent,
    }


# --- Run experiments ---
# view_radius: 1 → 3x3 patch (9 cells)
#              2 → 5x5 patch (25 cells)
#              3 → 7x7 patch (49 cells)
#             -1 → full 10x10 grid (100 cells)

configs  = [1, 2, 3, -1]
results  = {}

for r in configs:
    results[r] = train_ppo(r, total_steps=200_000)

## 4. Learning Curves

In [ ]:
def smooth(data, window=30):
    """Moving average smoothing."""
    if len(data) < window:
        return data
    return np.convolve(data, np.ones(window)/window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PPO on Stochastic Gridworld: Effect of Local View Size', fontsize=14, fontweight='bold')

colors = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']
labels_nice = {'1': '3×3 patch (r=1)', '2': '5×5 patch (r=2)',
               '3': '7×7 patch (r=3)', '-1': 'Full grid (10×10)'}

# --- Left: smoothed episode returns ---
ax = axes[0]
for i, r in enumerate(configs):
    res = results[r]
    s   = smooth(res['returns'], window=40)
    xs  = np.linspace(0, 200_000, len(s))
    ax.plot(xs, s, color=colors[i], linewidth=2, label=labels_nice[str(r)])

ax.axhline(1.0,  color='green', linestyle='--', alpha=0.3, linewidth=1)
ax.axhline(-0.5, color='red',   linestyle='--', alpha=0.3, linewidth=1)
ax.set_xlabel('Environment Steps')
ax.set_ylabel('Episode Return (smoothed)')
ax.set_title('Learning Curves')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 200_000)

# --- Right: success rate over time ---
ax2 = axes[1]
for i, r in enumerate(configs):
    res = results[r]
    successes = [1 if ret > 0.5 else 0 for ret in res['returns']]
    s = smooth(successes, window=50)
    xs = np.linspace(0, 200_000, len(s))
    ax2.plot(xs, s * 100, color=colors[i], linewidth=2, label=labels_nice[str(r)])

ax2.set_xlabel('Environment Steps')
ax2.set_ylabel('Success Rate (%)')
ax2.set_title('Success Rate (reaching goal)')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 200_000)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved learning_curves.png')

## 5. Final Performance Summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

bar_labels, bar_sr, bar_ret = [], [], []
for r in configs:
    res = results[r]
    tail = res['returns'][-100:]  # last 100 episodes
    sr   = np.mean([1 if x > 0.5 else 0 for x in tail]) * 100
    avg  = np.mean(tail)
    bar_labels.append(labels_nice[str(r)])
    bar_sr.append(sr)
    bar_ret.append(avg)
    print(f"{labels_nice[str(r)]:30s}  success={sr:5.1f}%  avg_return={avg:.3f}  obs_size={results[r]['obs_size']}")

x = np.arange(len(bar_labels))
w = 0.4
bars1 = ax.bar(x - w/2, bar_sr,  width=w, color=colors, alpha=0.85, label='Success Rate (%)')
ax2b  = ax.twinx()
bars2 = ax2b.bar(x + w/2, bar_ret, width=w, color=colors, alpha=0.45, label='Avg Return')

ax.set_ylabel('Success Rate (%)')
ax2b.set_ylabel('Avg Return')
ax.set_xticks(x)
ax.set_xticklabels(bar_labels, fontsize=9)
ax.set_title('Final Performance (last 100 episodes)', fontweight='bold')
ax.set_ylim(0, 110)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=9)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('final_performance.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Visualize Agent Behavior on a New Grid

Watch the trained agent navigate a **fresh grid it has never seen**.

In [ ]:
def rollout_episode(agent, view_radius, max_steps=150):
    """Run one full episode, record trajectory."""
    env = StochasticGridworld(view_radius=view_radius)
    obs = env.reset()
    trajectory = [env.pos]
    grid_snapshot = env.grid.copy()
    total_reward = 0

    for _ in range(max_steps):
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        with torch.no_grad():
            logits, _ = agent.net(obs_t)
            action = torch.argmax(logits).item()  # greedy
        obs, reward, done = env.step(action)
        trajectory.append(env.pos)
        total_reward += reward
        if done:
            break

    return grid_snapshot, trajectory, total_reward, env.goal


def plot_trajectory(ax, grid, trajectory, goal, title, color):
    G = grid.shape[0]
    cmap = ListedColormap(['#f8f9fa', '#2c3e50'])  # empty, wall
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, origin='upper')

    # Draw grid lines
    for i in range(G+1):
        ax.axhline(i-0.5, color='#bdc3c7', linewidth=0.5)
        ax.axvline(i-0.5, color='#bdc3c7', linewidth=0.5)

    # Draw trajectory
    if len(trajectory) > 1:
        rows = [p[0] for p in trajectory]
        cols = [p[1] for p in trajectory]
        ax.plot(cols, rows, '-', color=color, linewidth=2, alpha=0.8)
        ax.plot(cols[0], rows[0], 'o', color='#27ae60', markersize=9, zorder=5, label='start')
        ax.plot(cols[-1], rows[-1], 's', color=color, markersize=9, zorder=5)

    # Goal marker
    ax.plot(goal[1], goal[0], '*', color='gold', markersize=14, zorder=6,
            markeredgecolor='darkorange', markeredgewidth=1)

    reached = trajectory[-1] == goal
    status  = '✓ reached goal' if reached else '✗ did not reach'
    ax.set_title(f'{title}\n{status}  ({len(trajectory)} steps)', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])


# Plot trajectories for each view radius on fresh grids
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Agent Trajectories on Unseen Grids (greedy policy)', fontsize=13, fontweight='bold')

torch.manual_seed(99); np.random.seed(99); random.seed(99)

for i, r in enumerate(configs):
    grid, traj, total_r, goal = rollout_episode(results[r]['agent'], r)
    plot_trajectory(axes[i], grid, traj, goal,
                    title=labels_nice[str(r)],
                    color=colors[i])

# Legend
legend_elements = [
    mpatches.Patch(color='#2c3e50', label='Wall'),
    mpatches.Patch(color='#f8f9fa', label='Empty'),
    plt.Line2D([0],[0], marker='*', color='w', markerfacecolor='gold',
               markersize=12, markeredgecolor='darkorange', label='Goal'),
    plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='#27ae60',
               markersize=9, label='Start'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), fontsize=9)

plt.tight_layout()
plt.savefig('trajectories.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Interactive: Try Any View Radius

Pick a view radius, train a fresh agent, and watch it navigate.

In [ ]:
def quick_experiment(view_radius, total_steps=150_000):
    """
    Train a fresh agent with the given view radius and plot its behavior.
    Usage: quick_experiment(view_radius=2)
           quick_experiment(view_radius=-1)  # full grid
    """
    res = train_ppo(view_radius, total_steps=total_steps)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Quick Experiment: view_radius={res["label"]}  (obs_size={res["obs_size"]})',
                 fontsize=13, fontweight='bold')

    # Learning curve
    ax = axes[0]
    s = smooth(res['returns'], window=30)
    ax.plot(s, color='#2980b9', linewidth=2)
    ax.set_title('Episode Returns'); ax.set_xlabel('Episode'); ax.set_ylabel('Return')
    ax.grid(True, alpha=0.3)

    # Success rate
    ax = axes[1]
    successes = [1 if r > 0.5 else 0 for r in res['returns']]
    s = smooth(successes, window=50)
    ax.plot(s * 100, color='#27ae60', linewidth=2)
    ax.set_title('Success Rate'); ax.set_xlabel('Episode'); ax.set_ylabel('%')
    ax.set_ylim(0, 105); ax.grid(True, alpha=0.3)

    # Trajectory
    torch.manual_seed(7); np.random.seed(7); random.seed(7)
    grid, traj, total_r, goal = rollout_episode(res['agent'], view_radius)
    plot_trajectory(axes[2], grid, traj, goal, title='Greedy trajectory on fresh grid',
                    color='#e74c3c')

    tail_sr = np.mean([1 if r > 0.5 else 0 for r in res['returns'][-100:]]) * 100
    print(f'\nFinal 100-ep success rate: {tail_sr:.1f}%')

    plt.tight_layout()
    plt.show()
    return res


# ----- Try it yourself! -----
# Change the number below and re-run this cell.
#   1  = 3x3 local view (minimal context)
#   2  = 5x5 local view
#   3  = 7x7 local view
#   4  = 9x9 local view (almost full grid)
#  -1  = full 10x10 grid (oracle view)

my_result = quick_experiment(view_radius=2)

## Key Takeaways

| View | What the agent sees | What it learns |
|------|--------------------|-|
| **r=1 (3×3)** | Immediate neighbors only | Reactive: avoid walls, bias toward goal direction. May get stuck in local traps. |
| **r=2 (5×5)** | Small neighborhood | Can plan 2 steps ahead, handles most obstacles well. |
| **r=3 (7×7)** | Large neighborhood | Near-sighted path planning, competitive with full view on 10×10. |
| **Full grid** | Everything | Can plan globally, but the larger input makes learning slightly slower. |

**Why the sweet spot isn't always `full`:**  
A larger observation isn't free — the network has more to learn from it, and early in training the signal is noisy.
On small grids, r=2 or r=3 often converges just as fast as full because the local structure is sufficient to navigate.

**This is the core insight:** PPO learns a *generalizable behavioral strategy*, not a memorized path.
Even r=1 succeeds — it just learns "if there's a wall ahead, turn; aim toward goal corner" as a heuristic.
Tabular methods (Q-learning, SARSA) would need a separate Q-table entry for *every possible grid configuration* — exponentially infeasible.
